# 5. Transfer Learning for CNNs

In this notebook we load a small datasets that contains pictures of dolphins and elephants. We classify the images using CNNs and compare two approaches to see what works better:
1. Training a CNN from scratch.
2. Finetuning a pretrained ResNet.

In [2]:
import torch
from torchvision import transforms
import torchvision
import torch.nn as nn
import torch.nn.functional as F
import torchvision.datasets as datasets

torch.manual_seed(0)

Let's load our data and have a look at the shape of some images:

In [3]:
dataset = datasets.ImageFolder(root='./data/animals')
for i, data in enumerate(dataset):
    print(data)
    if i == 5:
        break

(<PIL.Image.Image image mode=RGB size=300x179 at 0x7F8A62855690>, 0)
(<PIL.Image.Image image mode=RGB size=300x179 at 0x7F8A62854310>, 0)
(<PIL.Image.Image image mode=RGB size=300x166 at 0x7F8A62855690>, 0)
(<PIL.Image.Image image mode=RGB size=300x259 at 0x7F8A62854310>, 0)
(<PIL.Image.Image image mode=RGB size=300x225 at 0x7F8A62855690>, 0)
(<PIL.Image.Image image mode=RGB size=300x277 at 0x7F8A62854310>, 0)


We see that the pictures have all `width=300` but a varying height. To use them in transfer learning they need to have the standard shape of size `(224, 224)`, which is the data format of ImageNet (on which most pretrained models are trained on).  

To get them into this shape, we first define a transformation that increases the image height to 224 (this will also increase the width) and then take the 224 pixel center square of the picture:

In [4]:
image_transforms = transforms.Compose([
             transforms.Resize(size=224),
             transforms.CenterCrop(size=224),
             transforms.ToTensor(),
             transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225]) # standard normalization for transfer learning
    ])

With this transformation, we now load all images from the disk.

Next, we split the data into train and test and define the data loaders that loads the data from disk.

In [5]:
data = datasets.ImageFolder(root='./data/animals', transform=image_transforms)
train_set, test_set = torch.utils.data.random_split(data, [100, 29])

batch_size = 10

trainloader = torch.utils.data.DataLoader(train_set, batch_size=batch_size,
                                          shuffle=True)
testloader = torch.utils.data.DataLoader(test_set, batch_size=29,
                                         shuffle=False)

## Tasks:
### Task 1.
Train a CNN from scratch to identify the object on the image (dolphin or elephant). For this, use the same CNN architecture as in Task 3 of the notebook from last week `04_CNNs.ipynb`. To make this work, here are a few things you need to change:
1. You need to change the input size of the fully-connected layer to match the new image dimension.
2. You need to change the output dimension of the fully-connected layer to classify only two classes instead of ten.

Used the dataloader to load the data (see cell above), which allows us to train our model in mini-batches (aka "mini-batch gradient descent"). We already did this last week, so you can check there how this works. 

Train for 20 epochs on the train data and afterwards compute the accuracy on the test data.

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


In [15]:
import torch.optim as optim
def train(model, optimizer, train_loader, test_loader, epochs=20):
    model.to(device)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_accs = [], []

    for epoch in range(epochs):
        # ── Training ──
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        avg_loss = running_loss / len(train_loader.dataset)
        train_losses.append(avg_loss)

        # ── Evaluation ──
        model.eval()
        correct = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                preds = torch.argmax(model(images), dim=1)
                correct += (preds == labels).sum().item()

        acc = correct / len(test_loader.dataset)
        test_accs.append(acc)
        print(f"Epoch {epoch+1}/{epochs}  Loss: {avg_loss:.4f}  Test accuracy: {acc*100:.2f}%")

    return train_losses, test_accs

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 4, kernel_size=3, padding=0)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(4 * 111 * 111, 2)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

cnn = CNN()
optimizer = optim.SGD(cnn.parameters(), lr=0.001)
losses, accs = train(cnn, optimizer, trainloader, testloader, epochs=20)



Epoch 1/20  Loss: 0.7340  Test accuracy: 79.31%
Epoch 2/20  Loss: 0.4063  Test accuracy: 68.97%
Epoch 3/20  Loss: 0.4038  Test accuracy: 79.31%
Epoch 4/20  Loss: 0.2445  Test accuracy: 75.86%
Epoch 5/20  Loss: 0.1937  Test accuracy: 79.31%
Epoch 6/20  Loss: 0.1726  Test accuracy: 82.76%
Epoch 7/20  Loss: 0.1417  Test accuracy: 82.76%
Epoch 8/20  Loss: 0.1247  Test accuracy: 82.76%
Epoch 9/20  Loss: 0.1092  Test accuracy: 82.76%
Epoch 10/20  Loss: 0.1062  Test accuracy: 82.76%
Epoch 11/20  Loss: 0.0833  Test accuracy: 82.76%
Epoch 12/20  Loss: 0.0761  Test accuracy: 82.76%
Epoch 13/20  Loss: 0.0681  Test accuracy: 82.76%
Epoch 14/20  Loss: 0.0624  Test accuracy: 82.76%
Epoch 15/20  Loss: 0.0571  Test accuracy: 82.76%
Epoch 16/20  Loss: 0.0531  Test accuracy: 82.76%
Epoch 17/20  Loss: 0.0498  Test accuracy: 82.76%
Epoch 18/20  Loss: 0.0455  Test accuracy: 82.76%
Epoch 19/20  Loss: 0.0419  Test accuracy: 82.76%
Epoch 20/20  Loss: 0.0405  Test accuracy: 82.76%


### Task 2:
Instead of training a CNN from scratch, we now want to load a pretrained **ResNet18** model and re-train its last layer to do our classifcation task.
PyTorch has a [tutorial](https://pytorch.org/tutorials/beginner/transfer_learning_tutorial.html#convnet-as-fixed-feature-extractor) on transfer learning, which you can check to see how this works (note: its enough to read the section `ConvNet as fixed feature extractor`).

Train the last layer of the pre-trained RestNet model for 20 epochs and compare the results to task 1.

In [17]:
model_conv = torchvision.models.resnet18(weights='IMAGENET1K_V1')
for param in model_conv.parameters():
    param.requires_grad = False

# Parameters of newly constructed modules have requires_grad=True by default
num_ftrs = model_conv.fc.in_features
model_conv.fc = nn.Linear(num_ftrs, 2)

model_conv = model_conv.to(device)


optimizer_conv = optim.SGD(model_conv.fc.parameters(), lr=0.001, momentum=0.9)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /Users/huangzhengqiang/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100.0%


In [18]:
model_conv = train(model_conv, optimizer_conv, trainloader, testloader, epochs=20)

Epoch 1/20  Loss: 0.6185  Test accuracy: 96.55%
Epoch 2/20  Loss: 0.3302  Test accuracy: 100.00%
Epoch 3/20  Loss: 0.1840  Test accuracy: 100.00%
Epoch 4/20  Loss: 0.1606  Test accuracy: 100.00%
Epoch 5/20  Loss: 0.1260  Test accuracy: 100.00%
Epoch 6/20  Loss: 0.0908  Test accuracy: 100.00%
Epoch 7/20  Loss: 0.1564  Test accuracy: 100.00%
Epoch 8/20  Loss: 0.0949  Test accuracy: 100.00%
Epoch 9/20  Loss: 0.1374  Test accuracy: 100.00%
Epoch 10/20  Loss: 0.1146  Test accuracy: 100.00%
Epoch 11/20  Loss: 0.0718  Test accuracy: 100.00%
Epoch 12/20  Loss: 0.0569  Test accuracy: 100.00%
Epoch 13/20  Loss: 0.1212  Test accuracy: 100.00%
Epoch 14/20  Loss: 0.0617  Test accuracy: 100.00%
Epoch 15/20  Loss: 0.0907  Test accuracy: 100.00%
Epoch 16/20  Loss: 0.0572  Test accuracy: 100.00%
Epoch 17/20  Loss: 0.0601  Test accuracy: 100.00%
Epoch 18/20  Loss: 0.0771  Test accuracy: 100.00%
Epoch 19/20  Loss: 0.0902  Test accuracy: 100.00%
Epoch 20/20  Loss: 0.1300  Test accuracy: 100.00%


### Bonus Task: Object Detection with YOLOv5

Instead of classifying the entire image, we now want to **detect and locate** objects in the images using a pretrained YOLOv5 model.

First, install the `ultralytics` package: `!pip install ultralytics seaborn -q`

1. Load the pretrained YOLOv5s model using `torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)`.
2. Run inference on 5 **elephant** images from the `./data/animals` folder. (Note: YOLOv5 is trained on the COCO dataset which includes "elephant" but not "dolphin".)
3. Display the results using the model's built-in `.show()` or `.render()` method.
4. Look at the detected classes and bounding boxes — does YOLOv5 detect the elephants correctly?
5. Compare: What is the difference between the **classification** task from Tasks 1–2 and the **object detection** task here?